In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

model = ChatOpenAI(model="gpt-5.6-luna")

# 초기 시스템 메시지
messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

response = model.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

안녕하세요, 개스톤. 초대해 주셔서 고마워요. 하지만 오늘 저녁은 사양할게요. 저는 조용히 책을 읽으며 시간을 보내고 싶답니다. 책 속 세상은 정말 멋지거든요.


## OutputParser: 모델 출력을 다음 단계가 쓰기 좋은 값으로 바꾸기

채팅 모델의 `invoke()` 결과는 보통 단순 문자열이 아니라 내용과 메타데이터를 함께 담은 `AIMessage`다. **OutputParser**는 이 모델 출력을 문자열, JSON, 리스트, Pydantic 객체 등 애플리케이션이 사용할 형태로 변환하는 `Runnable`(함수처럼 입력을 받아 출력을 반환하는 실행 가능한 객체)이다.

### `StrOutputParser`

- 입력: `AIMessage` 같은 모델 출력
- 출력: 메시지의 텍스트 내용인 `str`
- 용도: 화면 표시, 저장, 문자열 후처리처럼 메타데이터가 필요 없는 경우
- 스트리밍: 모델의 텍스트 청크를 그대로 전달할 수 있다.

```text
ChatModel.invoke(...) → AIMessage(content="...")
StrOutputParser      → "..."
```

In [4]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = model.invoke(messages)
parser.invoke(result)

'안녕하세요, 개스톤. 초대는 고맙지만 사양할게요. 오늘은 읽고 싶은 책이 있어서요. 저녁은 마을의 다른 아가씨들과 함께 드시는 게 어떨까요?'

## LCEL (LangChain Expression Language)

LCEL은 프롬프트, 모델, 파서처럼 **`Runnable` 인터페이스를 따르는 구성 요소를 선언적으로 조합하는 문법**이다. 파이프 연산자 `|`의 왼쪽 출력이 오른쪽 입력으로 전달된다.

```python
chain = prompt_template | model | parser
```

이 식은 함수 합성처럼 읽는다.

```text
dict → ChatPromptValue → AIMessage → str
       prompt_template    model       parser
```

### 공통 실행 메서드

| 메서드 | 의미 |
|---|---|
| `invoke(input)` | 입력 하나를 동기 실행 |
| `ainvoke(input)` | 입력 하나를 비동기 실행 |
| `batch(inputs)` / `abatch(inputs)` | 여러 입력을 처리 |
| `stream(input)` / `astream(input)` | 결과를 청크 단위로 받기 |

### 기억할 점

1. 체인을 만들 때 가장 먼저 **각 단계의 입출력 타입**을 확인한다. 서로 맞지 않으면 중간 변환이 필요하다.
2. `chain.invoke(...)`는 전체 파이프라인을 한 번에 실행한다. 중간값을 보고 싶다면 각 구성 요소를 따로 `invoke()`해 확인한다.
3. `|`는 데이터를 복사하는 기호가 아니라 실행 가능한 새 `Runnable`을 만드는 조합 연산자다.
4. 모든 단계가 스트리밍을 지원할 때 체인 전체에서도 자연스럽게 스트리밍할 수 있다.

In [5]:
chain = model | parser
chain.invoke(messages)

'안녕하세요, 개스톤. 초대해 주셔서 고마워요. 하지만 오늘 저녁은 사양할게요. 저는 조용히 책을 읽으며 시간을 보내고 싶거든요. 이해해 주시겠어요? 📖'

## ChatPromptTemplate: 역할이 있는 메시지 템플릿

`ChatPromptTemplate`은 문자열 하나가 아니라 `system`, `user`(또는 `human`), `ai` 같은 **역할별 메시지 목록**을 만든다. 반복되는 지시와 질문의 형태는 고정하고, 실행할 때 `{변수}`만 채울 수 있다.

```python
prompt = ChatPromptTemplate([
    ("system", "너는 {role}이다."),
    ("user", "{question}"),
])
prompt_value = prompt.invoke({"role": "튜터", "question": "LCEL이 뭐야?"})
```

### 입출력과 역할

- 입력: 템플릿 변수 이름과 값을 담은 `dict`
- 출력: 모델이 받을 메시지 목록을 감싼 `ChatPromptValue`
- `system`: 모델의 역할, 원칙, 제약 조건
- `user` / `human`: 사용자의 요청이나 현재 입력
- `ai`: 예시 응답이나 기존 대화의 AI 메시지

### 자주 쓰는 기능과 주의점

- 이전 대화처럼 길이가 가변인 메시지 목록은 `MessagesPlaceholder("history")`로 삽입한다.
- 자주 고정되는 변수는 `prompt.partial(...)`로 미리 채울 수 있다.
- 기본 템플릿 형식은 Python f-string 스타일이다. 글자 그대로의 중괄호가 필요하면 `{{`와 `}}`로 쓴다.
- 누락된 필수 변수는 실행 시 오류가 난다. 변수 이름을 작고 명확하게 유지한다.
- 외부에서 받은 신뢰할 수 없는 문자열을 `system` 템플릿 자체로 사용하지 않는다. 사용자 데이터는 변수 값으로 전달한다.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

response.pretty_print()

================================== Ai Message ==================================

안녕하세요, 개스톤. 초대해 주셔서 고마워요. 하지만 오늘 저녁은 사양할게요. 저는 조용히 책을 읽으며 시간을 보내고 싶답니다. 책 속 세상은 정말 멋지거든요.


## 전체 체인 연결

이제 세 구성 요소의 타입이 맞물린다.

| 단계 | 받는 값 | 내보내는 값 |
|---|---|---|
| `prompt_template` | 템플릿 변수 `dict` | `ChatPromptValue` |
| `model` | 메시지 기반 프롬프트 | `AIMessage` |
| `parser` | 모델 메시지 | `str` |

따라서 호출자는 템플릿 변수만 넘기고 최종 문자열을 받는다. 프롬프트 구성, 모델 호출, 결과 추출이 하나의 재사용 가능한 `Runnable`로 묶인 것이다.

In [8]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

'안녕하세요, 야수님. 초대해 주셔서 고마워요. 오늘 저녁이라면 기꺼이 함께할게요. 어떤 요리를 준비하셨는지 벌써 궁금한걸요?'

## 문자열 파싱과 구조화 출력의 차이

`StrOutputParser`는 텍스트만 꺼내므로 자유로운 대화 응답에 적합하다. 반면 아래의 `with_structured_output(Adlib)`은 모델 출력이 지정한 Pydantic 스키마를 따르도록 요청하고, 검증된 `Adlib` 객체를 반환한다.

| 필요 | 선택 | 최종 타입 |
|---|---|---|
| 답변 텍스트만 필요 | `StrOutputParser()` | `str` |
| 필드와 타입이 정해진 데이터 필요 | `with_structured_output(Model)` | Pydantic 모델 |

구조화 출력은 후속 코드에서 필드를 안전하게 참조할 때 유용하다. 그래도 값의 의미까지 자동으로 보장되지는 않으므로, 이 예시의 감정 강도처럼 범위가 중요한 필드는 Pydantic 제약 조건으로도 검증하는 것이 좋다.

In [9]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "개스톤",
    "activity": "저녁"
})

'안녕하세요, 개스톤. 초대해 주셔서 고맙지만, 오늘 저녁은 사양할게요. 저는 책을 읽으며 조용한 시간을 보내고 싶답니다. Baebele?'

In [10]:
from typing import Literal
from pydantic import BaseModel, Field

class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Adlib)
adlib_chain = prompt_template | structured_llm

adlib_chain.invoke({
    "story": "미녀와 야수",
    "character_a": "벨",
    "character_b": "개스톤",
    "activity": "저녁"
})

Adlib(answer='안녕하세요, 개스톤. 초대는 고맙지만 오늘 저녁은 사양할게요. 저는 조용히 책을 읽으며 시간을 보내는 걸 더 좋아한답니다. 부디 좋은 저녁 보내세요!', main_emotion='불쾌', main_emotion_intensity=0.25)